# Automation Suite

> **Authorized lab use only.** This notebook is an educational demonstration. Run it only with synthetic data or systems you own and are permitted to test.


In [ ]:
#!/usr/bin/env python3

import subprocess
import sys
import os
import json
import time
import threading
from datetime import datetime, timedelta
from typing import Dict, List, Optional, Callable, Any
import logging
from pathlib import Path
import csv
import re
from concurrent.futures import ThreadPoolExecutor

try:
    import schedule
except ImportError:
    schedule = None

try:
    import requests
except ImportError:
    requests = None

try:
    import smtplib
    from email.mime.text import MIMEText
    from email.mime.multipart import MIMEMultipart
except ImportError:
    smtplib = None

logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')

class AutomationSuite:
    def __init__(self):
        self.tasks = {}
        self.workflows = {}
        self.running_tasks = {}
        self.task_history = []
        self.config = {
            'max_threads': 5,
            'log_retention_days': 30,
            'default_timeout': 300
        }
        self.scheduler_running = False
        self.executor = ThreadPoolExecutor(max_workers=self.config['max_threads'])

    def install_dependencies(self):
        packages = [
            'schedule',
            'requests',
            'beautifulsoup4',
            'pandas',
            'openpyxl',
            'python-crontab',
            'psutil',
            'watchdog'
        ]

        print("Installing required packages...")
        for package in packages:
            try:
                subprocess.check_call([sys.executable, '-m', 'pip', 'install', package, '-q'])
                print(f"{package} installed")
            except subprocess.CalledProcessError:
                print(f"Failed to install {package}")

        # Re-import after installation
        try:
            global schedule, requests, smtplib
            import schedule
            import requests
            import smtplib
            from email.mime.text import MIMEText
            from email.mime.multipart import MIMEMultipart
            print("All modules imported successfully")
        except ImportError as e:
            print(f"Import error after installation: {e}")
            print("Please restart the runtime and try again")

    def create_task(self, name: str, func: Callable, args: tuple = (), kwargs: dict = None,
                   schedule_type: str = 'once', schedule_value: str = None,
                   timeout: int = None, retry_count: int = 0) -> str:
        if kwargs is None:
            kwargs = {}

        task_id = f"{name}_{datetime.now().strftime('%Y%m%d_%H%M%S')}"

        task = {
            'id': task_id,
            'name': name,
            'function': func,
            'args': args,
            'kwargs': kwargs,
            'schedule_type': schedule_type,
            'schedule_value': schedule_value,
            'timeout': timeout or self.config['default_timeout'],
            'retry_count': retry_count,
            'created_at': datetime.now(),
            'last_run': None,
            'next_run': None,
            'status': 'created',
            'result': None,
            'error': None
        }

        self.tasks[task_id] = task

        if schedule_type != 'once':
            self._schedule_task(task)

        logging.info(f"Created task: {name} ({task_id})")
        return task_id

    def _schedule_task(self, task: Dict):
        if schedule is None:
            print("Schedule module not available. Please install dependencies first.")
            return

        schedule_type = task['schedule_type']
        schedule_value = task['schedule_value']

        if schedule_type == 'interval':
            schedule.every(int(schedule_value)).seconds.do(self._run_task, task['id'])
        elif schedule_type == 'hourly':
            schedule.every().hour.do(self._run_task, task['id'])
        elif schedule_type == 'daily':
            if schedule_value:
                schedule.every().day.at(schedule_value).do(self._run_task, task['id'])
            else:
                schedule.every().day.do(self._run_task, task['id'])
        elif schedule_type == 'weekly':
            day = schedule_value.lower() if schedule_value else 'monday'
            getattr(schedule.every(), day).do(self._run_task, task['id'])
        elif schedule_type == 'cron':
            pass

    def _run_task(self, task_id: str):
        if task_id not in self.tasks:
            return

        task = self.tasks[task_id]

        if task_id in self.running_tasks:
            logging.warning(f"Task {task['name']} already running, skipping")
            return

        self.running_tasks[task_id] = True
        task['status'] = 'running'
        task['last_run'] = datetime.now()

        try:
            future = self.executor.submit(self._execute_task, task)
            result = future.result(timeout=task['timeout'])

            task['result'] = result
            task['status'] = 'completed'
            task['error'] = None

            logging.info(f"Task {task['name']} completed successfully")

        except Exception as e:
            task['error'] = str(e)
            task['status'] = 'failed'

            if task['retry_count'] > 0:
                task['retry_count'] -= 1
                logging.info(f"Task {task['name']} failed, retrying ({task['retry_count']} attempts left)")
                time.sleep(5)
                self._run_task(task_id)
            else:
                logging.error(f"Task {task['name']} failed: {e}")

        finally:
            if task_id in self.running_tasks:
                del self.running_tasks[task_id]

            self.task_history.append({
                'task_id': task_id,
                'name': task['name'],
                'status': task['status'],
                'start_time': task['last_run'],
                'end_time': datetime.now(),
                'result': task.get('result'),
                'error': task.get('error')
            })

    def _execute_task(self, task: Dict):
        return task['function'](*task['args'], **task['kwargs'])

    def run_task(self, task_id: str):
        if task_id in self.tasks:
            self._run_task(task_id)
        else:
            logging.error(f"Task {task_id} not found")

    def run_task_by_name(self, name: str):
        for task_id, task in self.tasks.items():
            if task['name'] == name:
                self._run_task(task_id)
                return
        logging.error(f"Task with name '{name}' not found")

    def start_scheduler(self):
        if schedule is None:
            print("Schedule module not available. Please install dependencies first.")
            return

        if self.scheduler_running:
            logging.warning("Scheduler already running")
            return

        self.scheduler_running = True

        def scheduler_loop():
            while self.scheduler_running:
                schedule.run_pending()
                time.sleep(1)

        scheduler_thread = threading.Thread(target=scheduler_loop, daemon=True)
        scheduler_thread.start()

        logging.info("Scheduler started")

    def stop_scheduler(self):
        self.scheduler_running = False
        logging.info("Scheduler stopped")

    def create_workflow(self, name: str, tasks: List[Dict]) -> str:
        workflow_id = f"{name}_{datetime.now().strftime('%Y%m%d_%H%M%S')}"

        workflow = {
            'id': workflow_id,
            'name': name,
            'tasks': tasks,
            'created_at': datetime.now(),
            'status': 'created'
        }

        self.workflows[workflow_id] = workflow
        logging.info(f"Created workflow: {name} ({workflow_id})")
        return workflow_id

    def run_workflow(self, workflow_id: str):
        if workflow_id not in self.workflows:
            logging.error(f"Workflow {workflow_id} not found")
            return

        workflow = self.workflows[workflow_id]
        workflow['status'] = 'running'

        try:
            for task_config in workflow['tasks']:
                if task_config['type'] == 'parallel':
                    self._run_parallel_tasks(task_config['tasks'])
                elif task_config['type'] == 'sequential':
                    self._run_sequential_tasks(task_config['tasks'])
                elif task_config['type'] == 'conditional':
                    self._run_conditional_task(task_config)
                else:
                    task_id = self.create_task(**task_config)
                    self.run_task(task_id)

            workflow['status'] = 'completed'
            logging.info(f"Workflow {workflow['name']} completed")

        except Exception as e:
            workflow['status'] = 'failed'
            logging.error(f"Workflow {workflow['name']} failed: {e}")

    def _run_parallel_tasks(self, tasks: List[Dict]):
        futures = []
        for task_config in tasks:
            task_id = self.create_task(**task_config)
            future = self.executor.submit(self._run_task, task_id)
            futures.append(future)

        for future in futures:
            future.result()

    def _run_sequential_tasks(self, tasks: List[Dict]):
        for task_config in tasks:
            task_id = self.create_task(**task_config)
            self.run_task(task_id)

    def _run_conditional_task(self, task_config: Dict):
        condition = task_config['condition']
        if condition():
            task_id = self.create_task(**task_config['task'])
            self.run_task(task_id)
        elif 'else_task' in task_config:
            task_id = self.create_task(**task_config['else_task'])
            self.run_task(task_id)

    def get_task_status(self, task_id: str) -> Dict:
        if task_id in self.tasks:
            return {
                'id': task_id,
                'name': self.tasks[task_id]['name'],
                'status': self.tasks[task_id]['status'],
                'last_run': self.tasks[task_id]['last_run'],
                'result': self.tasks[task_id]['result'],
                'error': self.tasks[task_id]['error']
            }
        return None

    def list_tasks(self) -> List[Dict]:
        return [
            {
                'id': task_id,
                'name': task['name'],
                'status': task['status'],
                'schedule_type': task['schedule_type'],
                'last_run': task['last_run'],
                'next_run': task['next_run']
            }
            for task_id, task in self.tasks.items()
        ]

    def delete_task(self, task_id: str):
        if task_id in self.tasks:
            del self.tasks[task_id]
            logging.info(f"Deleted task {task_id}")
        else:
            logging.error(f"Task {task_id} not found")

    def export_tasks(self, filename: str = 'tasks_export.json'):
        export_data = {
            'tasks': self.tasks,
            'workflows': self.workflows,
            'task_history': self.task_history,
            'exported_at': datetime.now().isoformat()
        }

        with open(filename, 'w') as f:
            json.dump(export_data, f, indent=2, default=str)

        logging.info(f"Tasks exported to {filename}")

    def import_tasks(self, filename: str):
        try:
            with open(filename, 'r') as f:
                data = json.load(f)

            self.tasks.update(data.get('tasks', {}))
            self.workflows.update(data.get('workflows', {}))
            self.task_history.extend(data.get('task_history', []))

            logging.info(f"Tasks imported from {filename}")

        except Exception as e:
            logging.error(f"Failed to import tasks: {e}")

class AutomationTasks:

    @staticmethod
    def file_backup(source_dir: str, dest_dir: str, pattern: str = "*"):
        import shutil
        import glob

        source_path = Path(source_dir)
        dest_path = Path(dest_dir)
        dest_path.mkdir(parents=True, exist_ok=True)

        files = glob.glob(str(source_path / pattern))
        backup_count = 0

        for file_path in files:
            file_name = os.path.basename(file_path)
            timestamp = datetime.now().strftime('%Y%m%d_%H%M%S')
            backup_name = f"{timestamp}_{file_name}"

            shutil.copy2(file_path, dest_path / backup_name)
            backup_count += 1

        return f"Backed up {backup_count} files"

    @staticmethod
    def system_cleanup(temp_dirs: List[str] = None, max_age_days: int = 7):
        import shutil

        if temp_dirs is None:
            temp_dirs = ['/tmp', '/var/tmp']

        cleanup_count = 0
        cutoff_date = datetime.now() - timedelta(days=max_age_days)

        for temp_dir in temp_dirs:
            if os.path.exists(temp_dir):
                for root, dirs, files in os.walk(temp_dir):
                    for file in files:
                        file_path = os.path.join(root, file)
                        try:
                            if os.path.getmtime(file_path) < cutoff_date.timestamp():
                                os.remove(file_path)
                                cleanup_count += 1
                        except:
                            pass

        return f"Cleaned up {cleanup_count} files"

    @staticmethod
    def web_monitor(url: str, expected_status: int = 200, expected_content: str = None):
        if requests is None:
            return "Requests module not available. Please install dependencies first."

        try:
            response = requests.get(url, timeout=30)

            if response.status_code != expected_status:
                return f"Status check failed: {response.status_code}"

            if expected_content and expected_content not in response.text:
                return f"Content check failed: '{expected_content}' not found"

            return "Website is healthy"

        except Exception as e:
            return f"Website check failed: {str(e)}"

    @staticmethod
    def database_backup(db_config: Dict):
        db_type = db_config.get('type', 'mysql')
        host = db_config.get('host', 'localhost')
        port = db_config.get('port', 3306)
        username = db_config.get('username')
        password = db_config.get('password')
        database = db_config.get('database')

        timestamp = datetime.now().strftime('%Y%m%d_%H%M%S')
        backup_file = f"{database}_backup_{timestamp}.sql"

        if db_type == 'mysql':
            cmd = f"mysqldump -h {host} -P {port} -u {username} -p{password} {database} > {backup_file}"
        elif db_type == 'postgresql':
            cmd = f"pg_dump -h {host} -p {port} -U {username} {database} > {backup_file}"
        else:
            return f"Unsupported database type: {db_type}"

        try:
            subprocess.run(cmd, shell=True, check=True)
            return f"Database backup created: {backup_file}"
        except subprocess.CalledProcessError as e:
            return f"Database backup failed: {e}"

    @staticmethod
    def log_analysis(log_file: str, pattern: str, action: str = 'count'):
        if not os.path.exists(log_file):
            return f"Log file not found: {log_file}"

        matches = []
        with open(log_file, 'r') as f:
            for line_num, line in enumerate(f, 1):
                if re.search(pattern, line):
                    matches.append((line_num, line.strip()))

        if action == 'count':
            return f"Found {len(matches)} matches"
        elif action == 'extract':
            return matches
        else:
            return f"Unknown action: {action}"

    @staticmethod
    def email_report(smtp_config: Dict, recipients: List[str], subject: str, body: str):
        if smtplib is None:
            return "Email modules not available. Please install dependencies first."

        try:
            from email.mime.text import MIMEText
            from email.mime.multipart import MIMEMultipart

            msg = MIMEMultipart()
            msg['From'] = smtp_config['from']
            msg['To'] = ', '.join(recipients)
            msg['Subject'] = subject

            msg.attach(MIMEText(body, 'plain'))

            server = smtplib.SMTP(smtp_config['host'], smtp_config['port'])
            server.starttls()
            server.login(smtp_config['username'], smtp_config['password'])

            server.send_message(msg)
            server.quit()

            return f"Email sent to {len(recipients)} recipients"

        except Exception as e:
            return f"Email failed: {str(e)}"

    @staticmethod
    def csv_processor(input_file: str, output_file: str, operations: List[Dict]):
        try:
            import pandas as pd

            df = pd.read_csv(input_file)

            for operation in operations:
                op_type = operation['type']

                if op_type == 'filter':
                    df = df.query(operation['condition'])
                elif op_type == 'sort':
                    df = df.sort_values(operation['columns'])
                elif op_type == 'group':
                    df = df.groupby(operation['columns']).agg(operation['aggregation'])
                elif op_type == 'transform':
                    df[operation['column']] = df[operation['column']].apply(operation['function'])

            df.to_csv(output_file, index=False)
            return f"CSV processed: {len(df)} rows written to {output_file}"

        except Exception as e:
            return f"CSV processing failed: {str(e)}"

def create_sample_automation():
    automation = AutomationSuite()
    automation.install_dependencies()

    task1 = automation.create_task(
        name="System Health Check",
        func=AutomationTasks.web_monitor,
        args=("https://httpbin.org/status/200",),
        schedule_type="interval",
        schedule_value="60"
    )

    task2 = automation.create_task(
        name="Log Analysis",
        func=AutomationTasks.log_analysis,
        args=("/var/log/syslog", "error"),
        schedule_type="daily",
        schedule_value="09:00"
    )

    workflow_tasks = [
        {
            'name': 'Check Website',
            'func': AutomationTasks.web_monitor,
            'args': ('https://httpbin.org/status/200',)
        },
        {
            'name': 'Backup Files',
            'func': AutomationTasks.file_backup,
            'args': ('/tmp', '/backup', '*.txt')
        }
    ]

    workflow1 = automation.create_workflow("Daily Maintenance", workflow_tasks)

    return automation

def main():
    automation = create_sample_automation()

    print("Automation Suite Started")
    print("Available commands:")
    print("- automation.start_scheduler()")
    print("- automation.run_task_by_name('System Health Check')")
    print("- automation.run_workflow(workflow_id)")
    print("- automation.list_tasks()")
    print("- automation.get_task_status(task_id)")

    automation.start_scheduler()

    try:
        while True:
            time.sleep(10)
            print(f"Active tasks: {len(automation.running_tasks)}")
    except KeyboardInterrupt:
        automation.stop_scheduler()
        print("Automation stopped")

if __name__ == "__main__":
    main()